In [ ]:
# --- CELL 1: SETUP & REPRODUCIBILITY ---
!pip install -q timm albumentations kagglehub

import os
import cv2
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Modern device-agnostic mixed precision (Resolves PyTorch deprecation warnings)
from torch.amp import autocast, GradScaler 

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score, accuracy_score
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import timm
import kagglehub
from tqdm.notebook import tqdm

def seed_everything(seed=42):
    """
    Ensures strict mathematical reproducibility.
    Peer-reviewed journals require exact reproduction of reported metrics.
    """
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
print("[SUCCESS] Environment Setup Complete. Libraries Installed and Seed Set.")

In [ ]:
# --- CELL 2: RESEARCH METRICS (ECE & SENSITIVITY/SPECIFICITY) ---

def calculate_ece(probs, labels, n_bins=10):
    """
    Expected Calibration Error (ECE).
    Measures the trustworthiness of the model. Lower ECE means the model's 
    confidence scores accurately reflect its true probability of being correct.
    """
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers, bin_uppers = bin_boundaries[:-1], bin_boundaries[1:]
    
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = predictions == labels
    
    ece = np.zeros(1)
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = in_bin.astype(float).mean()
        if prop_in_bin.item() > 0:
            accuracy_in_bin = accuracies[in_bin].astype(float).mean()
            avg_confidence_in_bin = confidences[in_bin].mean()
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    return ece[0]

def get_sensitivity_specificity(y_true, y_pred, num_classes=5):
    """
    Calculates Sensitivity (Recall), Specificity, and F1-Score per class.
    Essential for medical journals to prove the model minimizes False Negatives.
    """
    cm = confusion_matrix(y_true, y_pred)
    metrics = {}
    for i in range(num_classes):
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - tp - fp - fn
        
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
        
        metrics[f'Class_{i}'] = {'Sens': sensitivity, 'Spec': specificity, 'F1': f1}
    return metrics
print("[SUCCESS] Research Metrics (ECE, Sensitivity, Specificity) loaded.")

In [ ]:
# --- CELL 3: CONFIGURATION & DATA MAPPING (SWIN VERSION) ---
class Config:
    seed = 42
    img_size = 224        # Swin Tiny uses a 224x224 patch layout
    num_classes = 5       
    batch_size = 32       
    epochs = 10           
    n_folds = 5           
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # UPDATED: Changed model name for saving weights and OOF CSV
    model_name = 'swin_tiny' 
    
    # Base dataset path mapping
    base_path = kagglehub.dataset_download("ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy")
    train_dir = os.path.join(base_path, "augmented_resized_V2", "train")
    csv_save_path = "dataset_map.csv"

def create_dataset_map():
    if os.path.exists(Config.csv_save_path):
        return pd.read_csv(Config.csv_save_path)

    data = []
    classes = sorted([d for d in os.listdir(Config.train_dir) if os.path.isdir(os.path.join(Config.train_dir, d))])
    
    for class_name in classes:
        class_path = os.path.join(Config.train_dir, class_name)
        try: label = int(class_name)
        except ValueError:
            mapping = {'No_DR': 0, 'Mild': 1, 'Moderate': 2, 'Severe': 3, 'Proliferate_DR': 4}
            label = mapping.get(class_name, -1)
        if label == -1: continue

        files = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        for f in files: data.append({'image_path': os.path.join(class_path, f), 'label': label})
    
    df = pd.DataFrame(data)
    skf = StratifiedKFold(n_splits=Config.n_folds, shuffle=True, random_state=Config.seed)
    df['fold'] = -1
    for fold, (_, val_idx) in enumerate(skf.split(df, df['label'])):
        df.loc[val_idx, 'fold'] = fold
        
    df.to_csv(Config.csv_save_path, index=False)
    return df

df = create_dataset_map()
print(f"[SUCCESS] Swin Dataset mapped successfully. Total images: {len(df)}")

In [ ]:
# --- CELL 4: DATASET & ARCHITECTURE (SWIN VERSION) ---
class DRDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row['image_path'])
        if image is None: image = np.zeros((Config.img_size, Config.img_size, 3), dtype=np.uint8)
        else: image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform: image = self.transform(image=image)['image']
        return image, torch.tensor(row['label'], dtype=torch.long)

def get_transforms(data='train'):
    if data == 'train':
        return A.Compose([
            A.Resize(Config.img_size, Config.img_size),
            A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
            A.CLAHE(clip_limit=4.0, p=0.7), 
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(Config.img_size, Config.img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

class DRModel(nn.Module):
    def __init__(self, model_name, num_classes=5, pretrained=True):
        super(DRModel, self).__init__()
        
        # UPDATED: Load the Swin Transformer from timm
        # We use 'swin_tiny_patch4_window7_224' as it perfectly matches our 224 image size
        self.backbone = timm.create_model('swin_tiny_patch4_window7_224', pretrained=pretrained, num_classes=0)
        
        self.n_features = self.backbone.num_features
        self.drop = nn.Dropout(p=0.3)
        self.fc = nn.Linear(self.n_features, num_classes)

    def forward(self, x):
        return self.fc(self.drop(self.backbone(x)))

print("[SUCCESS] DRDataset Loader and Swin Transformer Architecture defined.")

In [ ]:
# --- CELL 5: LOSS & TRAINING UTILITIES ---
class FocalLoss(nn.Module):
    """
    Addresses class imbalance by down-weighting well-classified examples 
    and focusing training on hard, minority classes (like Proliferative DR).
    """
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha, self.gamma, self.reduction = alpha, gamma, reduction

    def forward(self, inputs, targets):
        CE_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-CE_loss)
        F_loss = self.alpha * (1-pt)**self.gamma * CE_loss
        if self.reduction == 'mean': return torch.mean(F_loss)
        return F_loss

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    pbar = tqdm(loader, desc="Training", leave=False)
    
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with autocast('cuda'): 
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        
    return running_loss / len(loader)

@torch.no_grad()
def valid_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    preds_all, labels_all = [], []
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        running_loss += loss.item()
        preds_all.append(outputs.softmax(1).cpu().numpy())
        labels_all.append(labels.cpu().numpy())
        
    preds_all = np.concatenate(preds_all)
    labels_all = np.concatenate(labels_all)
    predictions = np.argmax(preds_all, axis=1)
    
    qwk = cohen_kappa_score(labels_all, predictions, weights='quadratic')
    return running_loss / len(loader), qwk, preds_all, labels_all
print("[SUCCESS] Focal Loss and Epoch Training functions defined.")

In [ ]:
# --- CELL 6: 5-FOLD RUNNER WITH OOF EXTRACTION ---
def run_5fold_training_and_extract_oof():
    df = pd.read_csv(Config.csv_save_path)
    fold_results = []
    
    # Array to store the probabilities for the whole dataset for the Ensemble
    oof_probabilities = np.zeros((len(df), Config.num_classes))
    
    print(f"Starting Phase 1 Training: {Config.model_name}")
    
    for fold in range(Config.n_folds):
        print(f"\n{'='*40}\n FOLD {fold} \n{'='*40}")
        
        val_idx = df[df['fold'] == fold].index
        
        train_df = df.iloc[~df.index.isin(val_idx)].reset_index(drop=True)
        valid_df = df.iloc[val_idx].reset_index(drop=True)
        
        train_loader = DataLoader(DRDataset(train_df, transform=get_transforms('train')), 
                                  batch_size=Config.batch_size, shuffle=True, num_workers=2)
        valid_loader = DataLoader(DRDataset(valid_df, transform=get_transforms('valid')), 
                                  batch_size=Config.batch_size, shuffle=False, num_workers=2)
        
        model = DRModel(Config.model_name, num_classes=Config.num_classes).to(Config.device)
        optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
        criterion = FocalLoss(gamma=2.0)
        scaler = GradScaler('cuda') 
        
        best_qwk, best_ece = 0.0, 0.0
        best_fold_probs = None 
        
        for epoch in range(Config.epochs):
            train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, Config.device)
            val_loss, qwk, probs, labels = valid_one_epoch(model, valid_loader, criterion, Config.device)
            
            if qwk > best_qwk:
                best_qwk = qwk
                best_ece = calculate_ece(probs, labels)
                best_fold_probs = probs 
                
                torch.save(model.state_dict(), f"{Config.model_name}_fold{fold}_best.pth")
                print(f"Epoch {epoch+1} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | QWK: {qwk:.4f}  <- SAVED")
            else:
                print(f"Epoch {epoch+1} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | QWK: {qwk:.4f}")
        
        # Map the best probabilities back to the main array
        oof_probabilities[val_idx] = best_fold_probs
        fold_results.append({'fold': fold, 'qwk': best_qwk, 'ece': best_ece})
        
    # --- Phase 1 OOF Saving ---
    for i in range(Config.num_classes):
        df[f'{Config.model_name}_prob_{i}'] = oof_probabilities[:, i]
    
    oof_filename = f"oof_predictions_{Config.model_name}.csv"
    df.to_csv(oof_filename, index=False)
    print(f"\n[SUCCESS] Phase 1 Complete. OOF predictions saved to: {oof_filename}")
    print("\n" + "="*50)
    qwk_scores = [res['qwk'] for res in fold_results]
    print(f"FINAL QWK (Mean ± Std): {np.mean(qwk_scores):.4f} ± {np.std(qwk_scores):.4f}")

run_5fold_training_and_extract_oof()